In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import warnings

# --- 1. Настройка шрифтов и качества (строго по твоему списку) ---
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'font.size': 14,            # Основной шрифт
    'axes.titlesize': 16,       # Заголовки подграфиков
    'axes.labelsize': 14,       # Подписи осей
    'xtick.labelsize': 14,      # Деления по X
    'ytick.labelsize': 14,      # Деления по Y
    'legend.fontsize': 14,      # Шрифт в легенде
    'figure.dpi': 600,
    'savefig.dpi': 600
})

FILE_PATH = "/content/drive/MyDrive/Полином/задание (1).xlsx"

# --- 2. Вспомогательные функции ---
def load_data_raw(path):
    try:
        df_raw = pd.read_excel(path)
    except FileNotFoundError:
        print(f"Error: The file '{path}' was not found.")
        return None

    df = df_raw.iloc[3:, :].reset_index(drop=True)
    df = df.iloc[:, 1:]
    df.columns = [
        'exp_dead_f', 'exp_dead_m', 'exp_alive_f', 'exp_alive_m',
        'ctrl_dead_f', 'ctrl_dead_m', 'ctrl_alive_f', 'ctrl_alive_m'
    ]

    def to_float(x):
        try:
            if pd.isna(x) or x == '': return 0.0
            if isinstance(x, str): x = x.replace(',', '.')
            return float(x)
        except: return 0.0

    target_cols = ['exp_alive_f', 'exp_alive_m', 'ctrl_alive_f', 'ctrl_alive_m']
    for col in target_cols:
        df[col] = df[col].apply(to_float)

    df['age'] = np.arange(len(df))
    return df

def set_odd_xticks(ax, x_data):
    # Используем 14 шрифт для делений
    ticks = [i for i in range(int(np.min(x_data)), int(np.max(x_data)) + 1) if i % 2 == 1]
    ax.set_xticks(ticks)
    ax.set_xticklabels(ticks)

# --- 3. Подготовка и отрисовка ---
df_clean = load_data_raw(FILE_PATH)

if df_clean is not None:
    comparisons = [
        ("Females: Experimental vs Control", 'exp_alive_f', 'ctrl_alive_f'),
        ("Males: Experimental vs Control", 'exp_alive_m', 'ctrl_alive_m'),
        ("Experimental Group: Females vs Males", 'exp_alive_f', 'exp_alive_m'),
        ("Control Group: Females vs Males", 'ctrl_alive_f', 'ctrl_alive_m')
    ]

    window_length = 7
    polyorder = 2

    # Пропорции постера как в предыдущем коде
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    axes = axes.flatten()

    for i, (title, col1, col2) in enumerate(comparisons):
        mask = (df_clean[col1] > 0) | (df_clean[col2] > 0)
        last_idx = df_clean[mask].index[-1]

        x = df_clean['age'].values[:last_idx + 1]
        v1 = df_clean[col1].values[:last_idx + 1]
        v2 = df_clean[col2].values[:last_idx + 1]

        y_diff = v1 - v2
        w = min(window_length, len(y_diff))
        if w % 2 == 0: w = max(3, w - 1)
        y_diff_smooth = savgol_filter(y_diff, w, polyorder) if w > polyorder else y_diff

        # Замыкаем кривые на нуле
        x = np.append(x, x[-1] + 1)
        y_diff = np.append(y_diff, 0.0)
        y_diff_smooth = np.append(y_diff_smooth, 0.0)

        ax = axes[i]
        ax.plot(x, y_diff, 'o-', color='#8A2BE2', linewidth=3, markersize=8, label='Original Difference')
        ax.plot(x, y_diff_smooth, 'o-', color='#e6005c', linewidth=3, markersize=8, label='Smoothed Difference')

        ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
        ax.grid(True, linestyle=':', alpha=0.6)

        # Настройка заголовка: Полужирный и размер 16 (согласно rcParams)
        ax.set_title(title, fontweight='bold', pad=15)

        # Подписи осей (размер подхватится из rcParams = 14)
        ax.set_xlabel('Age (days)')
        ax.set_ylabel('Survival Difference (%)')

        set_odd_xticks(ax, x)
        ax.legend(loc='best')

        # Стилизация осей (ось X на нуле)
        ax.spines['bottom'].set_position('zero')
        ax.spines['bottom'].set_linewidth(1.5)
        ax.spines['top'].set_linewidth(0.5)
        ax.spines['left'].set_linewidth(0.5)
        ax.spines['right'].set_linewidth(0.5)

        ax.tick_params(axis='x', direction='out', pad=5)
        for label in ax.get_xticklabels():
            label.set_verticalalignment('top')

    # Оптимизация расположения элементов
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # Добавляем финальные штрихи по рамкам
    for ax in axes:
        y_min = ax.get_ylim()[0]
        ax.axhline(y=y_min, color='black', linewidth=0.5, clip_on=False)

    plt.savefig('survival_poster_final_v2.png', dpi=600, bbox_inches='tight')
    plt.show()

In [ ]:
!pip install lifelines

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from lifelines import KaplanMeierFitter
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 12,
    'figure.dpi': 600,
    'savefig.dpi': 600
})

FILE_PATH = "/content/drive/MyDrive/Полином/задание (1).xlsx"

df = pd.read_excel(FILE_PATH)
df = df.iloc[:, 1:]

def is_number(x):
    if isinstance(x, (int, float, np.float64, np.int64)):
        return True
    if isinstance(x, str):
        try:
            float(x.replace(',', '.'))
            return True
        except ValueError:
            return False
    return False

df.columns = [
    'exp_dead_f', 'exp_dead_m', 'exp_alive_f', 'exp_alive_m',
    'ctrl_dead_f', 'ctrl_dead_m', 'ctrl_alive_f', 'ctrl_alive_m'
]

cols_alive = ['exp_alive_f', 'exp_alive_m', 'ctrl_alive_f', 'ctrl_alive_m']

mask = df[cols_alive].map(is_number).all(axis=1)
df = df[mask].reset_index(drop=True)
df['age'] = np.arange(len(df))

for col in cols_alive:
    df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

def prep_lifelines_data(df, alive_col, label):
    durations = []
    events = []
    last_alive = 100.0

    for i, row in df.iterrows():
        age = row['age']
        alive = row[alive_col]

        n_died = last_alive - alive

        if n_died > 0:
            durations.extend([age] * int(round(n_died)))
            events.extend([1] * int(round(n_died)))
            last_alive = alive
        elif n_died < 0:
            # Прыжок вверх — просто обновляем last_alive, не добавляем события
            last_alive = alive

    if last_alive > 0:
        durations.extend([df['age'].max()] * int(round(last_alive)))
        events.extend([0] * int(round(last_alive)))

    return pd.DataFrame({'time': durations, 'event': events, 'group': label})

comparisons = [
    ("Females: Experimental vs Control", 'exp_alive_f', 'ctrl_alive_f', 'Experimental', 'Control'),
    ("Males: Experimental vs Control", 'exp_alive_m', 'ctrl_alive_m', 'Experimental', 'Control'),
    ("Experimental Group: Females vs Males", 'exp_alive_f', 'exp_alive_m', 'Females', 'Males'),
    ("Control Group: Females vs Males", 'ctrl_alive_f', 'ctrl_alive_m', 'Females', 'Males')
]

colors = {
    'Experimental': '#ff0000',
    'Control': '#8A2BE2',
    'Females': '#ff0000',
    'Males': '#8A2BE2'
}

fig, axes = plt.subplots(2, 2, figsize=(22, 16))
axes = axes.flatten()

for idx, (title, col1, col2, label1, label2) in enumerate(comparisons):
    ax = axes[idx]

    df1 = prep_lifelines_data(df, col1, label1)
    df2 = prep_lifelines_data(df, col2, label2)

    kmf = KaplanMeierFitter()
    legend_handles = []

    for data_df, label_name in zip([df1, df2], [label1, label2]):
        c = colors[label_name]

        kmf.fit(data_df['time'], data_df['event'], label=label_name)
        kmf.plot_survival_function(ax=ax, ci_show=True, color=c, linewidth=2.5)

        line_patch = plt.Line2D([0], [0], color=c, lw=2.5, label=f'{label_name} (KM Curve)')
        ci_patch = mpatches.Patch(color=c, alpha=0.3, label=f'{label_name} (95% CI)')
        legend_handles.extend([line_patch, ci_patch])

    ax.set_title(title, fontweight='bold', pad=15)
    ax.set_xlabel('Age (days)')
    ax.set_ylabel('Survival Probability')
    ax.grid(True, linestyle=':', alpha=0.7)

    ax.get_legend().remove()
    ax.legend(handles=legend_handles, loc='best')

    ax.set_ylim([0, 1.05])
    ax.set_xlim(left=0)

plt.tight_layout(pad=3.0)
plt.show()

In [ ]:
"""
ANALYSIS OF MORTALITY DYNAMICS: GAUSSIAN DECOMPOSITION OF SURVIVAL DIFFERENCES
================================================================================
Methodology:
  1. Raw survival difference ΔS(t) = S_group1(t) - S_group2(t)
  2. Savitzky–Golay smoothing (window=7, polyorder=2)
  3. Split into positive (ΔS > 0) and negative (ΔS < 0) envelopes
  4. For each envelope: iterative Gaussian fitting with BIC-based model selection
  5. Confidence intervals from covariance matrix of curve_fit
  6. Composite fit: total = pos_fit - neg_fit

Key design decisions:
  - BIC (not AIC) for component selection: stronger penalty at small n
  - Savitzky–Golay pre-smoothing (window=7, poly=2) before decomposition
  - CI for all parameters; components with CI(amplitude) crossing zero flagged
  - All groups processed by identical algorithm (no hardcoded switches)
  - Plots show RAW ΔS(t) as scatter points, smoothed signal as dashed line
  - Metrics tables (MSE / RMSE / R²) printed for both raw and smoothed signals
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from scipy.stats import norm
from scipy.interpolate import interp1d
import warnings
import logging

# ============================================================
# SETUP
# ============================================================
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger(__name__)

plt.rcParams.update({
    'font.size': 14, 'axes.titlesize': 14, 'axes.labelsize': 14,
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 10,
    'figure.dpi': 600, 'savefig.dpi': 600
})

# ============================================================
# CONFIGURATION
# ============================================================
FILE_PATH = "/content/drive/MyDrive/Полином/задание (1).xlsx"  # Change to your path in Colab
MAX_GAUSSIANS_PER_HALF = 8            # Upper bound for BIC search
CONFIDENCE_LEVEL = 0.95               # For parameter CI
Z_CRIT = norm.ppf(1 - (1 - CONFIDENCE_LEVEL) / 2)  # 1.96 for 95%

# Savitzky–Golay parameters
SAVGOL_WINDOW = 7
SAVGOL_POLYORDER = 2


# ============================================================
# 1. DATA LOADING
# ============================================================
def load_data(path):
    """Load and parse the survival data from Excel."""
    try:
        df_raw = pd.read_excel(path)
    except FileNotFoundError:
        log.error(f"File not found: {path}")
        return None

    df = df_raw.iloc[3:, :].reset_index(drop=True)
    df = df.iloc[:, 1:]
    df.columns = [
        'exp_dead_f', 'exp_dead_m', 'exp_alive_f', 'exp_alive_m',
        'ctrl_dead_f', 'ctrl_dead_m', 'ctrl_alive_f', 'ctrl_alive_m'
    ]

    def to_float(x):
        try:
            if pd.isna(x) or x == '':
                return 0.0
            if isinstance(x, str):
                x = x.replace(',', '.')
            return float(x)
        except (ValueError, TypeError):
            return 0.0

    for col in ['exp_alive_f', 'exp_alive_m', 'ctrl_alive_f', 'ctrl_alive_m']:
        df[col] = df[col].apply(to_float)

    df['age'] = np.arange(len(df))
    return df


# ============================================================
# 2. SAVITZKY–GOLAY SMOOTHING
# ============================================================
def smooth_savgol(y, window_length=SAVGOL_WINDOW, polyorder=SAVGOL_POLYORDER):
    """
    Apply Savitzky–Golay filter to a 1-D signal.
    Adjusts window to be odd and ≤ len(y).
    Returns original array unchanged if too short for filtering.
    """
    w = min(window_length, len(y))
    if w % 2 == 0:
        w = max(3, w - 1)
    if w > polyorder:
        return savgol_filter(y, w, polyorder)
    return y.copy()


# ============================================================
# 3. GAUSSIAN MODEL
# ============================================================
def single_gaussian(x, amp, mu, sigma):
    """Single Gaussian function. Amplitude is always positive here."""
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def multi_gaussian(x, *params):
    """Sum of N Gaussians. params = [amp1, mu1, sigma1, amp2, ...]"""
    n = len(params) // 3
    result = np.zeros_like(x, dtype=float)
    for i in range(n):
        result += single_gaussian(x, params[3*i], params[3*i+1], params[3*i+2])
    return result


# ============================================================
# 4. BIC-BASED MODEL SELECTION
# ============================================================
def compute_bic(n_points, n_params, rss):
    """
    Bayesian Information Criterion.
    BIC = n * ln(RSS/n) + k * ln(n)
    Lower is better. Penalizes complexity more than AIC at small n.
    """
    if rss <= 0 or n_points <= n_params:
        return np.inf
    return n_points * np.log(rss / n_points) + n_params * np.log(n_points)


def guess_initial_params(x, y, n_gauss):
    """
    Generate initial parameter guesses by iteratively finding residual peaks.
    Each Gaussian gets: [amplitude, center, width].
    """
    residual = y.copy()
    params = []
    x_range = x[-1] - x[0]

    for _ in range(n_gauss):
        idx = np.argmax(residual)
        amp = max(residual[idx], 0.01)
        mu = x[idx]
        sigma = max(1.0, x_range / (2 * n_gauss))
        params.extend([amp, mu, sigma])
        residual = residual - single_gaussian(x, amp, mu, sigma)
        residual = np.maximum(residual, 0)

    return params


def fit_n_gaussians(x, y, n_gauss, label=""):
    """
    Fit n Gaussians to positive data y.
    Returns dict with parameters, covariance, fit curve, MSE, BIC, CI.
    Returns None if fitting fails.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n_points = len(x)
    n_params = 3 * n_gauss
    x_range = x[-1] - x[0] if len(x) > 1 else 1.0

    if np.max(y) < 1e-9:
        return None

    p0 = guess_initial_params(x, y, n_gauss)

    sigma_min = 1.0
    lower, upper = [], []
    for k in range(n_gauss):
        lower.extend([0.0, x[0] - 1.0, sigma_min])
        upper.extend([np.max(y) * 2.0, x[-1] + 1.0, x_range * 1.5])

    try:
        popt, pcov = curve_fit(
            multi_gaussian, x, y, p0=p0,
            bounds=(lower, upper), maxfev=50000
        )
    except Exception as e:
        log.warning(f"    {label} n={n_gauss}: fitting failed ({e})")
        return None

    if np.any(np.isinf(pcov)) or np.any(np.isnan(pcov)):
        log.warning(f"    {label} n={n_gauss}: covariance matrix invalid")
        return None

    y_fit = multi_gaussian(x, *popt)
    rss = np.sum((y - y_fit) ** 2)
    mse = rss / n_points
    bic = compute_bic(n_points, n_params, rss)

    std_errors = np.sqrt(np.diag(pcov))
    ci_half = Z_CRIT * std_errors

    components = []
    for k in range(n_gauss):
        amp, mu, sigma = popt[3*k], popt[3*k+1], popt[3*k+2]
        se_amp, se_mu, se_sigma = std_errors[3*k], std_errors[3*k+1], std_errors[3*k+2]
        ci_amp = ci_half[3*k]
        ci_mu = ci_half[3*k+1]
        ci_sigma = ci_half[3*k+2]
        significant = (amp - ci_amp) > 0

        components.append({
            'amplitude': amp, 'se_amplitude': se_amp, 'ci_amplitude': ci_amp,
            'mu': mu, 'se_mu': se_mu, 'ci_mu': ci_mu,
            'sigma': sigma, 'se_sigma': se_sigma, 'ci_sigma': ci_sigma,
            'significant': significant
        })

    return {
        'n_gauss': n_gauss, 'popt': popt, 'pcov': pcov,
        'y_fit': y_fit, 'mse': mse, 'bic': bic,
        'components': components, 'n_params': n_params
    }


def find_optimal_n_bic(x, y, label="", max_n=MAX_GAUSSIANS_PER_HALF):
    """
    Find optimal number of Gaussians using BIC.
    """
    if np.max(y) < 1e-9:
        log.info(f"    {label}: no signal (all ≈ 0), n=0")
        return {}, 0

    threshold = np.max(y) * 0.01
    n_informative = np.sum(y > threshold)
    max_n_by_data = max(1, n_informative // 6)
    effective_max_n = min(max_n, max_n_by_data)

    log.info(f"    {label}: {n_informative} informative points → max {effective_max_n} Gaussians")

    results = {}
    best_bic = np.inf
    best_n = 0

    for n in range(1, effective_max_n + 1):
        res = fit_n_gaussians(x, y, n, label=label)

        if res is None:
            results[n] = None
            continue

        results[n] = res
        marker = ""
        if res['bic'] < best_bic:
            best_bic = res['bic']
            best_n = n
            marker = " ← best"

        log.info(f"    {label} n={n}: MSE={res['mse']:.4f}, BIC={res['bic']:.2f}{marker}")

    log.info(f"    {label}: optimal n={best_n} (BIC={best_bic:.2f})")
    return results, best_n


# ============================================================
# 5. FULL DECOMPOSITION (SPLIT POS/NEG)
# ============================================================
def decompose_survival_difference(x, y_diff, label=""):
    """
    Full decomposition pipeline:
    1. Split y_diff into positive and negative envelopes
    2. Fit each with BIC-selected number of Gaussians
    3. Combine: total = pos_fit - neg_fit
    4. Return all parameters with CI
    """
    y_pos = np.maximum(y_diff, 0.0)
    y_neg_abs = np.maximum(-y_diff, 0.0)

    log.info(f"  --- Positive envelope ---")
    res_pos, n_pos = find_optimal_n_bic(x, y_pos, label="POS")

    log.info(f"  --- Negative envelope ---")
    res_neg, n_neg = find_optimal_n_bic(x, y_neg_abs, label="NEG")

    if n_pos > 0 and res_pos.get(n_pos) is not None:
        best_pos = res_pos[n_pos]
        y_fit_pos = best_pos['y_fit']
    else:
        best_pos = None
        y_fit_pos = np.zeros_like(x)

    if n_neg > 0 and res_neg.get(n_neg) is not None:
        best_neg = res_neg[n_neg]
        y_fit_neg = best_neg['y_fit']
    else:
        best_neg = None
        y_fit_neg = np.zeros_like(x)

    y_fit_total = y_fit_pos - y_fit_neg
    mse_total = np.mean((y_diff - y_fit_total) ** 2)

    return {
        'n_pos': n_pos, 'n_neg': n_neg,
        'best_pos': best_pos, 'best_neg': best_neg,
        'y_fit_pos': y_fit_pos, 'y_fit_neg': -y_fit_neg,
        'y_fit_total': y_fit_total,
        'mse_total': mse_total,
        'res_pos': res_pos, 'res_neg': res_neg
    }


# ============================================================
# 6. METRICS (MSE / RMSE / R²)
# ============================================================
def compute_metrics(y_true, y_fit):
    """Compute MSE, RMSE, R²."""
    residuals = y_true - y_fit
    mse  = np.mean(residuals ** 2)
    rmse = np.sqrt(mse)
    var  = np.var(y_true)
    r2   = 1.0 - mse / var if var > 0 else np.nan
    return mse, rmse, r2


def r2_quality(r2):
    if np.isnan(r2):  return "n/a"
    if r2 >= 0.95:    return "excellent"
    if r2 >= 0.90:    return "good"
    if r2 >= 0.80:    return "acceptable"
    return "review model"


def print_metrics_table(title, y_fit, y_smooth, y_raw):
    """
    Print MSE / RMSE / R² table for both smoothed and raw signals.
    Format mirrors the second (assessment) script.
    """
    mse_s, rmse_s, r2_s = compute_metrics(y_smooth, y_fit)
    mse_r, rmse_r, r2_r = compute_metrics(y_raw,    y_fit)

    # Normalise RMSE by max absolute amplitude of the fit
    amp_max = np.max(np.abs(y_fit)) if np.max(np.abs(y_fit)) > 0 else 1.0

    print(f"\n  {'─'*70}")
    print(f"  {title}")
    print(f"  {'─'*70}")
    print(f"  {'Metric':<8} {'vs Smoothed':>14} {'vs Raw':>14}")
    print(f"  {'─'*40}")
    print(f"  {'MSE':<8} {mse_s:>14.4f} {mse_r:>14.4f}")
    print(f"  {'RMSE':<8} {rmse_s:>14.4f} {rmse_r:>14.4f}")
    print(f"  {'RMSE%':<8} {rmse_s/amp_max*100:>13.2f}% {rmse_r/amp_max*100:>13.2f}%")
    print(f"  {'Var':<8} {np.var(y_smooth):>14.4f} {np.var(y_raw):>14.4f}")
    print(f"  {'R²':<8} {r2_s:>14.4f} {r2_r:>14.4f}")
    print(f"  {'Quality':<8} {r2_quality(r2_s):>14} {r2_quality(r2_r):>14}")

    return mse_s, rmse_s, r2_s, mse_r, rmse_r, r2_r


# ============================================================
# 6.5 SUPERPOSITION CALCULATION
# ============================================================
def get_superposition_amplitude(t, fit_result):
    """Calculate the total model value (pos - neg) at a given point t."""
    val_pos = 0.0
    val_neg = 0.0
    if fit_result['best_pos'] is not None:
        val_pos = multi_gaussian(np.array([t]), *fit_result['best_pos']['popt'])[0]
    if fit_result['best_neg'] is not None:
        val_neg = multi_gaussian(np.array([t]), *fit_result['best_neg']['popt'])[0]
    return val_pos - val_neg


# ============================================================
# 7. REPORTING
# ============================================================
def print_parameters_table(fit_result, title):
    """Print formatted parameter table with CI and significance flags."""
    n_pos = fit_result['n_pos']
    n_neg = fit_result['n_neg']
    mse = fit_result['mse_total']

    print(f"\n{'─'*90}")
    print(f"  {title}")
    print(f"  {n_pos} positive + {n_neg} negative = {n_pos + n_neg} total components")
    print(f"  Total MSE = {mse:.4f}")
    print(f"{'─'*90}")
    print(f"  {'Comp':<7} {'Sign':<5} {'Amplitude':>11} {'± CI':>9} "
          f"{'μ (day)':>9} {'± CI':>8} {'σ':>8} {'± CI':>8} {'Signif.':>8}")
    print(f"  {'─'*83}")

    for sign_label, best, prefix in [('(+)', fit_result['best_pos'], '+G'),
                                      ('(−)', fit_result['best_neg'], '−G')]:
        if best is None:
            continue
        for k, comp in enumerate(best['components']):
            amp = comp['amplitude'] if prefix == '+G' else -comp['amplitude']
            sig_flag = '  ✓' if comp['significant'] else '  ✗'
            print(f"  {prefix}{k+1:<5} {sign_label:<5} "
                  f"{amp:>11.2f} {comp['ci_amplitude']:>8.2f} "
                  f"{comp['mu']:>9.2f} {comp['ci_mu']:>7.2f} "
                  f"{comp['sigma']:>8.2f} {comp['ci_sigma']:>7.2f} "
                  f"{sig_flag:>8}")

    print()


# ============================================================
# 8. VISUALIZATION
# ============================================================

def plot_decomposition(ax, x, y_raw, y_smooth, fit_result, title):
    """
    Plot decomposition with:
      - RAW ΔS(t) as scatter points (grey circles)
      - Total Gaussian fit as solid coloured line
      - Individual Gaussian components with fills
    """
    n_pos = fit_result['n_pos']
    n_neg = fit_result['n_neg']
    mse_s, rmse_s, r2_s = compute_metrics(y_smooth, fit_result['y_fit_total'])
    mse_r, rmse_r, r2_r = compute_metrics(y_raw,    fit_result['y_fit_total'])

    x_dense = np.linspace(x[0], x[-1], 1000)

    # ── RAW points ────────────────────────────────────────────────────────────
    ax.plot(x, y_raw, 'o', color='#555555', markersize=5, alpha=0.55,
            label='ΔS(t) raw', zorder=5)

    # ── Total fit on dense grid ───────────────────────────────────────────────
    f_total = interp1d(x, fit_result['y_fit_total'], kind='cubic',
                        bounds_error=False, fill_value='extrapolate')
    y_total_dense = f_total(x_dense)

    ax.plot(x_dense, y_total_dense, '-', color='#e6005c', linewidth=2.5,
            label=f'Total fit',
            zorder=6)

    # ── Individual positive Gaussians ─────────────────────────────────────────
    if fit_result['best_pos'] is not None:
        colors_p = plt.cm.Blues(np.linspace(0.4, 0.85, max(n_pos, 1)))
        popt = fit_result['best_pos']['popt']
        for k in range(n_pos):
            amp, mu, sigma = popt[3*k], popt[3*k+1], popt[3*k+2]
            y_k = single_gaussian(x_dense, amp, mu, sigma)
            ax.plot(x_dense, y_k, '-', color=colors_p[k], linewidth=1.5,
                    label=f'+G{k+1}')
            ax.fill_between(x_dense, 0, y_k, color=colors_p[k], alpha=0.12)

    # ── Individual negative Gaussians (inverted) ──────────────────────────────
    if fit_result['best_neg'] is not None:
        colors_n = plt.cm.Reds(np.linspace(0.4, 0.85, max(n_neg, 1)))
        popt = fit_result['best_neg']['popt']
        for k in range(n_neg):
            amp, mu, sigma = popt[3*k], popt[3*k+1], popt[3*k+2]
            y_k = -single_gaussian(x_dense, amp, mu, sigma)
            ax.plot(x_dense, y_k, '-', color=colors_n[k], linewidth=1.5,
                    label=f'−G{k+1}')
            ax.fill_between(x_dense, 0, y_k, color=colors_n[k], alpha=0.12)

    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.set_title(
        f'{title}',
        fontweight='bold', fontsize=14)
    ax.set_xlabel('Age (days)', fontsize=14)
    ax.set_ylabel('Survival Difference (%)', fontsize=14)
    ax.legend(loc='best', fontsize=9, ncol=2)


def plot_bic_curves(ax, fit_result, title):
    """Plot BIC vs number of Gaussians for pos and neg halves."""
    for res_dict, n_opt, color, label_prefix in [
        (fit_result['res_pos'], fit_result['n_pos'], '#2196F3', 'Positive'),
        (fit_result['res_neg'], fit_result['n_neg'], '#F44336', 'Negative')
    ]:
        ns = sorted([n for n in res_dict if n > 0 and res_dict[n] is not None])
        if not ns:
            continue
        bics = [res_dict[n]['bic'] for n in ns]
        ax.plot(ns, bics, 'o-', color=color, linewidth=2, markersize=6,
                label=f'{label_prefix} (opt={n_opt})')
        ax.axvline(n_opt, color=color, linestyle='--', alpha=0.5)

    ax.set_title(title, fontweight='bold', fontsize=14)
    ax.set_xlabel('Number of Gaussians', fontsize=14)
    ax.set_ylabel('BIC', fontsize=14)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend(fontsize=14)


# ============================================================
# 9. MAIN PIPELINE
# ============================================================
def main():
    df = load_data(FILE_PATH)
    if df is None:
        return

    comparisons = [
        ("Females: Experimental vs Control", 'exp_alive_f', 'ctrl_alive_f'),
        ("Males: Experimental vs Control",   'exp_alive_m', 'ctrl_alive_m'),
        ("Experimental Group: Females vs Males", 'exp_alive_f', 'exp_alive_m'),
        ("Control Group: Females vs Males",  'ctrl_alive_f', 'ctrl_alive_m')
    ]

    all_results = []

    for i, (title, col1, col2) in enumerate(comparisons):
        log.info(f"\n{'='*70}")
        log.info(f"[{i+1}] {title}")
        log.info(f"{'='*70}")

        mask = (df[col1] > 0) | (df[col2] > 0)
        last_idx = df[mask].index[-1]

        x = df['age'].values[:last_idx + 1].astype(float)

        # Keep both raw and smoothed signals
        y_diff_raw    = (df[col1].values[:last_idx + 1] -
                         df[col2].values[:last_idx + 1]).astype(float)

        # --- ЗАМЫКАНИЕ НА НОЛЬ ---
        x = np.append(x, x[-1] + 1.0)
        y_diff_raw = np.append(y_diff_raw, 0.0)
        # -------------------------

        y_diff_smooth = smooth_savgol(y_diff_raw)

        # Decomposition is performed on the smoothed signal
        fit_result = decompose_survival_difference(x, y_diff_smooth, label=title)

        log.info(f"\n  >>> RESULT: {fit_result['n_pos']}+ / {fit_result['n_neg']}− "
                 f"= {fit_result['n_pos'] + fit_result['n_neg']} total, "
                 f"MSE={fit_result['mse_total']:.4f}")

        all_results.append({
            'title': title,
            'x': x,
            'y_raw': y_diff_raw,
            'y_smooth': y_diff_smooth,
            'fit': fit_result
        })

    # ── Parameter tables ──────────────────────────────────────────────────────
    print("\n" + "="*90)
    print("  GAUSSIAN DECOMPOSITION: PARAMETER ESTIMATES WITH 95% CI")
    print("  BIC-based model selection | Savitzky–Golay smoothed (window=7, poly=2)")
    print("="*90)

    for r in all_results:
        print_parameters_table(r['fit'], r['title'])

    # ── Significant components only ───────────────────────────────────────────
    print("\n" + "="*90)
    print("  SIGNIFICANT COMPONENTS ONLY (for paper)")
    print("  Components where 95% CI for amplitude does NOT include zero")
    print("="*90)

    for r in all_results:
        fit = r['fit']
        title = r['title']
        sig_count = 0
        total_count = fit['n_pos'] + fit['n_neg']
        lines = []

        for sign_label, best, prefix, neg in [('(+)', fit['best_pos'], '+G', False),
                                               ('(−)', fit['best_neg'], '−G', True)]:
            if best is None:
                continue
            for k, comp in enumerate(best['components']):
                if comp['significant']:
                    sig_count += 1
                    amp = -comp['amplitude'] if neg else comp['amplitude']
                    lines.append(
                        f"  {prefix}{k+1:<5} {sign_label:<5} "
                        f"{amp:>11.2f} ± {comp['ci_amplitude']:>6.2f} "
                        f"{comp['mu']:>9.2f} ± {comp['ci_mu']:>5.2f} "
                        f"{comp['sigma']:>8.2f} ± {comp['ci_sigma']:>5.2f}"
                    )

        print(f"\n  {title}")
        print(f"  {sig_count} significant / {total_count} total components")
        print(f"  {'Comp':<7} {'Sign':<5} {'Amplitude':>11} {'± CI':>9} "
              f"{'μ (day)':>9} {'± CI':>8} {'σ':>8} {'± CI':>8}")
        print(f"  {'─'*70}")
        for line in lines:
            print(line)

    # ── Metrics tables (MSE / RMSE / R²) ─────────────────────────────────────
    print("\n\n" + "="*90)
    print("  GAUSSIAN FIT QUALITY: MSE / RMSE / R²")
    print("  R² = 1 - MSE / Var(y)  |  >0.95 excellent | >0.90 good | <0.80 review")
    print("  (s) = vs smoothed signal   (r) = vs raw signal")
    print("="*90)

    summary_rows = []

    for r in all_results:
        metrics = print_metrics_table(
            r['title'],
            r['fit']['y_fit_total'],
            r['y_smooth'],
            r['y_raw']
        )
        summary_rows.append((r['title'], *metrics))

    # ── Summary metrics table (paper-ready) ──────────────────────────────────
    print("\n\n" + "="*90)
    print("  SUMMARY TABLE (for paper)")
    print("="*90)
    print(f"  {'Comparison':<42} {'MSE(s)':>7} {'RMSE(s)':>8} {'R²(s)':>7} "
          f"{'MSE(r)':>7} {'RMSE(r)':>8} {'R²(r)':>7}  Quality(raw)")
    print(f"  {'─'*88}")

    for row in summary_rows:
        name, mse_s, rmse_s, r2_s, mse_r, rmse_r, r2_r = row
        short = name[:40]
        print(f"  {short:<42} {mse_s:>7.4f} {rmse_s:>8.4f} {r2_s:>7.4f} "
              f"{mse_r:>7.4f} {rmse_r:>8.4f} {r2_r:>7.4f}  {r2_quality(r2_r)}")

    # ── Superposition amplitudes at Gaussian centers ──────────────────────────
    print("\n\n" + "="*90)
    print("  SUPERPOSITION AMPLITUDES AT GAUSSIAN CENTERS (μ)")
    print("="*90)

    for r in all_results:
        fit = r['fit']
        print(f"\n  {r['title']}")
        print(f"  {'Comp':<7} {'Center (μ)':>12} | {'Superposition Amp (%)':>22}")
        print(f"  {'─'*47}")

        centers = []
        if fit['best_pos'] is not None:
            for k, comp in enumerate(fit['best_pos']['components']):
                centers.append((f"+G{k+1}", comp['mu']))
        if fit['best_neg'] is not None:
            for k, comp in enumerate(fit['best_neg']['components']):
                centers.append((f"−G{k+1}", comp['mu']))

        # Сортируем по времени (μ)
        centers.sort(key=lambda item: item[1])

        for name, mu in centers:
            total_amp = get_superposition_amplitude(mu, fit)
            print(f"  {name:<7} {mu:>12.2f} | {total_amp:>22.2f}")

    print()
    print("="*90)

    # ── BIC curves ────────────────────────────────────────────────────────────
    fig_bic, axes_bic = plt.subplots(2, 2, figsize=(16, 10))
    for i, r in enumerate(all_results):
        plot_bic_curves(axes_bic.flatten()[i], r['fit'], r['title'])
    fig_bic.suptitle('BIC vs Number of Gaussians (model selection)',
                     fontweight='bold', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.savefig('bic_curves.png', dpi=600, bbox_inches='tight')
    plt.show()

    # ── Main decomposition plots ──────────────────────────────────────────────
    fig_main, axes_main = plt.subplots(2, 2, figsize=(20, 14))
    for i, r in enumerate(all_results):
        plot_decomposition(
            axes_main.flatten()[i],
            r['x'], r['y_raw'], r['y_smooth'],
            r['fit'], r['title']
        )
    fig_main.suptitle(
        ''
        '',
        fontweight='bold', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.savefig('decomposition_smooth.png', dpi=600, bbox_inches='tight')
    plt.show()

    return all_results


# ============================================================
# RUN
# ============================================================
if __name__ == '__main__':
    results = main()

In [ ]:
"""
ANALYSIS OF MORTALITY DYNAMICS: GAUSSIAN DECOMPOSITION OF SURVIVAL DIFFERENCES
================================================================================
Methodology:
  1. Raw survival difference ΔS(t) = S_group1(t) - S_group2(t)
  2. Savitzky–Golay smoothing (window=7, polyorder=2)
  3. Split into positive (ΔS > 0) and negative (ΔS < 0) envelopes
  4. For each envelope: iterative Gaussian fitting with BIC-based model selection
  5. Confidence intervals from covariance matrix of curve_fit
  6. Composite fit: total = pos_fit - neg_fit

Key design decisions:
  - BIC (not AIC) for component selection: stronger penalty at small n
  - Savitzky–Golay pre-smoothing (window=7, poly=2) before decomposition
  - CI for all parameters; components with CI(amplitude) crossing zero flagged
  - All groups processed by identical algorithm (no hardcoded switches)
  - Plots show RAW ΔS(t) as scatter points, smoothed signal as dashed line
  - Metrics tables (MSE / RMSE / R²) printed for both raw and smoothed signals
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from scipy.stats import norm
from scipy.interpolate import interp1d
import warnings
import logging

# ============================================================
# SETUP
# ============================================================
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger(__name__)

plt.rcParams.update({
    'font.size': 14, 'axes.titlesize': 14, 'axes.labelsize': 14,
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 10,
    'figure.dpi': 600, 'savefig.dpi': 600
})

# ============================================================
# CONFIGURATION
# ============================================================
FILE_PATH = "/content/drive/MyDrive/Полином/задание (1).xlsx"  # Change to your path in Colab
MAX_GAUSSIANS_PER_HALF = 8            # Upper bound for BIC search
CONFIDENCE_LEVEL = 0.95               # For parameter CI
Z_CRIT = norm.ppf(1 - (1 - CONFIDENCE_LEVEL) / 2)  # 1.96 for 95%

# Savitzky–Golay parameters
SAVGOL_WINDOW = 7
SAVGOL_POLYORDER = 2


# ============================================================
# 1. DATA LOADING
# ============================================================
def load_data(path):
    """Load and parse the survival data from Excel."""
    try:
        df_raw = pd.read_excel(path)
    except FileNotFoundError:
        log.error(f"File not found: {path}")
        return None

    df = df_raw.iloc[3:, :].reset_index(drop=True)
    df = df.iloc[:, 1:]
    df.columns = [
        'exp_dead_f', 'exp_dead_m', 'exp_alive_f', 'exp_alive_m',
        'ctrl_dead_f', 'ctrl_dead_m', 'ctrl_alive_f', 'ctrl_alive_m'
    ]

    def to_float(x):
        try:
            if pd.isna(x) or x == '':
                return 0.0
            if isinstance(x, str):
                x = x.replace(',', '.')
            return float(x)
        except (ValueError, TypeError):
            return 0.0

    for col in ['exp_alive_f', 'exp_alive_m', 'ctrl_alive_f', 'ctrl_alive_m']:
        df[col] = df[col].apply(to_float)

    df['age'] = np.arange(len(df))
    return df


# ============================================================
# 2. SAVITZKY–GOLAY SMOOTHING
# ============================================================
def smooth_savgol(y, window_length=SAVGOL_WINDOW, polyorder=SAVGOL_POLYORDER):
    """
    Apply Savitzky–Golay filter to a 1-D signal.
    Adjusts window to be odd and ≤ len(y).
    Returns original array unchanged if too short for filtering.
    """
    w = min(window_length, len(y))
    if w % 2 == 0:
        w = max(3, w - 1)
    if w > polyorder:
        return savgol_filter(y, w, polyorder)
    return y.copy()


# ============================================================
# 3. GAUSSIAN MODEL
# ============================================================
def single_gaussian(x, amp, mu, sigma):
    """Single Gaussian function. Amplitude is always positive here."""
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def multi_gaussian(x, *params):
    """Sum of N Gaussians. params = [amp1, mu1, sigma1, amp2, ...]"""
    n = len(params) // 3
    result = np.zeros_like(x, dtype=float)
    for i in range(n):
        result += single_gaussian(x, params[3*i], params[3*i+1], params[3*i+2])
    return result


# ============================================================
# 4. BIC-BASED MODEL SELECTION
# ============================================================
def compute_bic(n_points, n_params, rss):
    """
    Bayesian Information Criterion.
    BIC = n * ln(RSS/n) + k * ln(n)
    Lower is better. Penalizes complexity more than AIC at small n.
    """
    if rss <= 0 or n_points <= n_params:
        return np.inf
    return n_points * np.log(rss / n_points) + n_params * np.log(n_points)


def guess_initial_params(x, y, n_gauss):
    """
    Generate initial parameter guesses by iteratively finding residual peaks.
    Each Gaussian gets: [amplitude, center, width].
    """
    residual = y.copy()
    params = []
    x_range = x[-1] - x[0]

    for _ in range(n_gauss):
        idx = np.argmax(residual)
        amp = max(residual[idx], 0.01)
        mu = x[idx]
        sigma = max(1.0, x_range / (2 * n_gauss))
        params.extend([amp, mu, sigma])
        residual = residual - single_gaussian(x, amp, mu, sigma)
        residual = np.maximum(residual, 0)

    return params


def fit_n_gaussians(x, y, n_gauss, label=""):
    """
    Fit n Gaussians to positive data y.
    Returns dict with parameters, covariance, fit curve, MSE, BIC, CI.
    Returns None if fitting fails.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n_points = len(x)
    n_params = 3 * n_gauss
    x_range = x[-1] - x[0] if len(x) > 1 else 1.0

    if np.max(y) < 1e-9:
        return None

    p0 = guess_initial_params(x, y, n_gauss)

    sigma_min = 1.0
    lower, upper = [], []
    for k in range(n_gauss):
        lower.extend([0.0, x[0] - 1.0, sigma_min])
        upper.extend([np.max(y) * 2.0, x[-1] + 1.0, x_range * 1.5])

    try:
        popt, pcov = curve_fit(
            multi_gaussian, x, y, p0=p0,
            bounds=(lower, upper), maxfev=50000
        )
    except Exception as e:
        log.warning(f"    {label} n={n_gauss}: fitting failed ({e})")
        return None

    if np.any(np.isinf(pcov)) or np.any(np.isnan(pcov)):
        log.warning(f"    {label} n={n_gauss}: covariance matrix invalid")
        return None

    y_fit = multi_gaussian(x, *popt)
    rss = np.sum((y - y_fit) ** 2)
    mse = rss / n_points
    bic = compute_bic(n_points, n_params, rss)

    std_errors = np.sqrt(np.diag(pcov))
    ci_half = Z_CRIT * std_errors

    components = []
    for k in range(n_gauss):
        amp, mu, sigma = popt[3*k], popt[3*k+1], popt[3*k+2]
        se_amp, se_mu, se_sigma = std_errors[3*k], std_errors[3*k+1], std_errors[3*k+2]
        ci_amp = ci_half[3*k]
        ci_mu = ci_half[3*k+1]
        ci_sigma = ci_half[3*k+2]
        significant = (amp - ci_amp) > 0

        components.append({
            'amplitude': amp, 'se_amplitude': se_amp, 'ci_amplitude': ci_amp,
            'mu': mu, 'se_mu': se_mu, 'ci_mu': ci_mu,
            'sigma': sigma, 'se_sigma': se_sigma, 'ci_sigma': ci_sigma,
            'significant': significant
        })

    return {
        'n_gauss': n_gauss, 'popt': popt, 'pcov': pcov,
        'y_fit': y_fit, 'mse': mse, 'bic': bic,
        'components': components, 'n_params': n_params
    }


def find_optimal_n_bic(x, y, label="", max_n=MAX_GAUSSIANS_PER_HALF):
    """
    Find optimal number of Gaussians using BIC.
    """
    if np.max(y) < 1e-9:
        log.info(f"    {label}: no signal (all ≈ 0), n=0")
        return {}, 0

    threshold = np.max(y) * 0.01
    n_informative = np.sum(y > threshold)
    max_n_by_data = max(1, n_informative // 6)
    effective_max_n = min(max_n, max_n_by_data)

    log.info(f"    {label}: {n_informative} informative points → max {effective_max_n} Gaussians")

    results = {}
    best_bic = np.inf
    best_n = 0

    for n in range(1, effective_max_n + 1):
        res = fit_n_gaussians(x, y, n, label=label)

        if res is None:
            results[n] = None
            continue

        results[n] = res
        marker = ""
        if res['bic'] < best_bic:
            best_bic = res['bic']
            best_n = n
            marker = " ← best"

        log.info(f"    {label} n={n}: MSE={res['mse']:.4f}, BIC={res['bic']:.2f}{marker}")

    log.info(f"    {label}: optimal n={best_n} (BIC={best_bic:.2f})")
    return results, best_n


# ============================================================
# 5. FULL DECOMPOSITION (SPLIT POS/NEG)
# ============================================================
def decompose_survival_difference(x, y_diff, label=""):
    """
    Full decomposition pipeline:
    1. Split y_diff into positive and negative envelopes
    2. Fit each with BIC-selected number of Gaussians
    3. Combine: total = pos_fit - neg_fit
    4. Return all parameters with CI
    """
    y_pos = np.maximum(y_diff, 0.0)
    y_neg_abs = np.maximum(-y_diff, 0.0)

    log.info(f"  --- Positive envelope ---")
    res_pos, n_pos = find_optimal_n_bic(x, y_pos, label="POS")

    log.info(f"  --- Negative envelope ---")
    res_neg, n_neg = find_optimal_n_bic(x, y_neg_abs, label="NEG")

    if n_pos > 0 and res_pos.get(n_pos) is not None:
        best_pos = res_pos[n_pos]
        y_fit_pos = best_pos['y_fit']
    else:
        best_pos = None
        y_fit_pos = np.zeros_like(x)

    if n_neg > 0 and res_neg.get(n_neg) is not None:
        best_neg = res_neg[n_neg]
        y_fit_neg = best_neg['y_fit']
    else:
        best_neg = None
        y_fit_neg = np.zeros_like(x)

    y_fit_total = y_fit_pos - y_fit_neg
    mse_total = np.mean((y_diff - y_fit_total) ** 2)

    return {
        'n_pos': n_pos, 'n_neg': n_neg,
        'best_pos': best_pos, 'best_neg': best_neg,
        'y_fit_pos': y_fit_pos, 'y_fit_neg': -y_fit_neg,
        'y_fit_total': y_fit_total,
        'mse_total': mse_total,
        'res_pos': res_pos, 'res_neg': res_neg
    }


# ============================================================
# 6. METRICS (MSE / RMSE / R²)
# ============================================================
def compute_metrics(y_true, y_fit):
    """Compute MSE, RMSE, R²."""
    residuals = y_true - y_fit
    mse  = np.mean(residuals ** 2)
    rmse = np.sqrt(mse)
    var  = np.var(y_true)
    r2   = 1.0 - mse / var if var > 0 else np.nan
    return mse, rmse, r2


def r2_quality(r2):
    if np.isnan(r2):  return "n/a"
    if r2 >= 0.95:    return "excellent"
    if r2 >= 0.90:    return "good"
    if r2 >= 0.80:    return "acceptable"
    return "review model"


def print_metrics_table(title, y_fit, y_smooth, y_raw):
    """
    Print MSE / RMSE / R² table for both smoothed and raw signals.
    """
    mse_s, rmse_s, r2_s = compute_metrics(y_smooth, y_fit)
    mse_r, rmse_r, r2_r = compute_metrics(y_raw,    y_fit)

    amp_max = np.max(np.abs(y_fit)) if np.max(np.abs(y_fit)) > 0 else 1.0

    print(f"\n  {'─'*70}")
    print(f"  {title}")
    print(f"  {'─'*70}")
    print(f"  {'Metric':<8} {'vs Smoothed':>14} {'vs Raw':>14}")
    print(f"  {'─'*40}")
    print(f"  {'MSE':<8} {mse_s:>14.4f} {mse_r:>14.4f}")
    print(f"  {'RMSE':<8} {rmse_s:>14.4f} {rmse_r:>14.4f}")
    print(f"  {'RMSE%':<8} {rmse_s/amp_max*100:>13.2f}% {rmse_r/amp_max*100:>13.2f}%")
    print(f"  {'Var':<8} {np.var(y_smooth):>14.4f} {np.var(y_raw):>14.4f}")
    print(f"  {'R²':<8} {r2_s:>14.4f} {r2_r:>14.4f}")
    print(f"  {'Quality':<8} {r2_quality(r2_s):>14} {r2_quality(r2_r):>14}")

    return mse_s, rmse_s, r2_s, mse_r, rmse_r, r2_r


# ============================================================
# 6.5 SUPERPOSITION CALCULATION
# ============================================================
def get_superposition_amplitude(t, fit_result):
    """Calculate the total model value (pos - neg) at a given point t."""
    val_pos = 0.0
    val_neg = 0.0
    if fit_result['best_pos'] is not None:
        val_pos = multi_gaussian(np.array([t]), *fit_result['best_pos']['popt'])[0]
    if fit_result['best_neg'] is not None:
        val_neg = multi_gaussian(np.array([t]), *fit_result['best_neg']['popt'])[0]
    return val_pos - val_neg


# ============================================================
# 7. REPORTING
# ============================================================
def print_parameters_table(fit_result, title):
    """Print formatted parameter table with CI and significance flags."""
    n_pos = fit_result['n_pos']
    n_neg = fit_result['n_neg']
    mse = fit_result['mse_total']

    print(f"\n{'─'*90}")
    print(f"  {title}")
    print(f"  {n_pos} positive + {n_neg} negative = {n_pos + n_neg} total components")
    print(f"  Total MSE = {mse:.4f}")
    print(f"{'─'*90}")
    print(f"  {'Comp':<7} {'Sign':<5} {'Amplitude':>11} {'± CI':>9} "
          f"{'μ (day)':>9} {'± CI':>8} {'σ':>8} {'± CI':>8} {'Signif.':>8}")
    print(f"  {'─'*83}")

    for sign_label, best, prefix in [('(+)', fit_result['best_pos'], '+G'),
                                      ('(−)', fit_result['best_neg'], '−G')]:
        if best is None:
            continue
        for k, comp in enumerate(best['components']):
            amp = comp['amplitude'] if prefix == '+G' else -comp['amplitude']
            sig_flag = '  ✓' if comp['significant'] else '  ✗'
            print(f"  {prefix}{k+1:<5} {sign_label:<5} "
                  f"{amp:>11.2f} {comp['ci_amplitude']:>8.2f} "
                  f"{comp['mu']:>9.2f} {comp['ci_mu']:>7.2f} "
                  f"{comp['sigma']:>8.2f} {comp['ci_sigma']:>7.2f} "
                  f"{sig_flag:>8}")

    print()


# ============================================================
# 8. VISUALIZATION
# ============================================================

def plot_decomposition(ax, x, y_raw, y_smooth, fit_result, title):
    """
    Plot decomposition with:
      - RAW ΔS(t) as scatter points
      - Total Gaussian fit as solid coloured line
      - Individual Gaussian components with fills
      - Black vertical lines and points precisely at Gaussian centers (μ)
    """
    n_pos = fit_result['n_pos']
    n_neg = fit_result['n_neg']
    mse_s, rmse_s, r2_s = compute_metrics(y_smooth, fit_result['y_fit_total'])
    mse_r, rmse_r, r2_r = compute_metrics(y_raw,    fit_result['y_fit_total'])

    x_dense = np.linspace(x[0], x[-1], 1000)

    # ── RAW points ────────────────────────────────────────────────────────────
    ax.plot(x, y_raw, 'o', color='#555555', markersize=5, alpha=0.55,
            label='ΔS(t) raw', zorder=5)

    # ── Total fit on dense grid ───────────────────────────────────────────────
    f_total = interp1d(x, fit_result['y_fit_total'], kind='cubic',
                        bounds_error=False, fill_value='extrapolate')
    y_total_dense = f_total(x_dense)

    ax.plot(x_dense, y_total_dense, '-', color='#e6005c', linewidth=2.5,
            label=f'Total fit  R²(smooth)={r2_s:.3f}  R²(raw)={r2_r:.3f}',
            zorder=6)

    # ── Individual positive Gaussians ─────────────────────────────────────────
    if fit_result['best_pos'] is not None:
        colors_p = plt.cm.Blues(np.linspace(0.4, 0.85, max(n_pos, 1)))
        popt = fit_result['best_pos']['popt']
        for k in range(n_pos):
            amp, mu, sigma = popt[3*k], popt[3*k+1], popt[3*k+2]
            y_k = single_gaussian(x_dense, amp, mu, sigma)
            ax.plot(x_dense, y_k, '-', color=colors_p[k], linewidth=1.5,
                    label=f'+G{k+1}')
            ax.fill_between(x_dense, 0, y_k, color=colors_p[k], alpha=0.12)

    # ── Individual negative Gaussians (inverted) ──────────────────────────────
    if fit_result['best_neg'] is not None:
        colors_n = plt.cm.Reds(np.linspace(0.4, 0.85, max(n_neg, 1)))
        popt = fit_result['best_neg']['popt']
        for k in range(n_neg):
            amp, mu, sigma = popt[3*k], popt[3*k+1], popt[3*k+2]
            y_k = -single_gaussian(x_dense, amp, mu, sigma)
            ax.plot(x_dense, y_k, '-', color=colors_n[k], linewidth=1.5,
                    label=f'−G{k+1}')
            ax.fill_between(x_dense, 0, y_k, color=colors_n[k], alpha=0.12)

    # ── Peaks on the plot (Black lines and dots at Gaussian centers) ──────────
    mus = []
    if fit_result['best_pos'] is not None:
        mus.extend([comp['mu'] for comp in fit_result['best_pos']['components']])
    if fit_result['best_neg'] is not None:
        mus.extend([comp['mu'] for comp in fit_result['best_neg']['components']])

    for mu in mus:
        amp = get_superposition_amplitude(mu, fit_result)
        ax.vlines(x=mu, ymin=0, ymax=amp, color='black', linestyle='-', linewidth=1.5, zorder=7)
        ax.plot(mu, amp, 'o', color='black', markersize=6, zorder=8)

    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.set_title(
        f'{title}\n({n_pos}+ / {n_neg}− = {n_pos+n_neg} components, BIC-selected)',
        fontweight='bold', fontsize=14)
    ax.set_xlabel('Age (days)', fontsize=14)
    ax.set_ylabel('Survival Difference (%)', fontsize=14)
    ax.legend(loc='best', fontsize=9, ncol=2)


def plot_bic_curves(ax, fit_result, title):
    """Plot BIC vs number of Gaussians for pos and neg halves."""
    for res_dict, n_opt, color, label_prefix in [
        (fit_result['res_pos'], fit_result['n_pos'], '#2196F3', 'Positive'),
        (fit_result['res_neg'], fit_result['n_neg'], '#F44336', 'Negative')
    ]:
        ns = sorted([n for n in res_dict if n > 0 and res_dict[n] is not None])
        if not ns:
            continue
        bics = [res_dict[n]['bic'] for n in ns]
        ax.plot(ns, bics, 'o-', color=color, linewidth=2, markersize=6,
                label=f'{label_prefix} (opt={n_opt})')
        ax.axvline(n_opt, color=color, linestyle='--', alpha=0.5)

    ax.set_title(title, fontweight='bold', fontsize=14)
    ax.set_xlabel('Number of Gaussians', fontsize=14)
    ax.set_ylabel('BIC', fontsize=14)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend(fontsize=14)


# ============================================================
# 9. MAIN PIPELINE
# ============================================================
def main():
    df = load_data(FILE_PATH)
    if df is None:
        return

    comparisons = [
        ("Females: Experimental vs Control", 'exp_alive_f', 'ctrl_alive_f'),
        ("Males: Experimental vs Control",   'exp_alive_m', 'ctrl_alive_m'),
        ("Experimental Group: Females vs Males", 'exp_alive_f', 'exp_alive_m'),
        ("Control Group: Females vs Males",  'ctrl_alive_f', 'ctrl_alive_m')
    ]

    all_results = []

    for i, (title, col1, col2) in enumerate(comparisons):
        log.info(f"\n{'='*70}")
        log.info(f"[{i+1}] {title}")
        log.info(f"{'='*70}")

        mask = (df[col1] > 0) | (df[col2] > 0)
        last_idx = df[mask].index[-1]

        x = df['age'].values[:last_idx + 1].astype(float)

        y_diff_raw    = (df[col1].values[:last_idx + 1] -
                         df[col2].values[:last_idx + 1]).astype(float)

        # --- ЗАМЫКАНИЕ НА НОЛЬ ---
        x = np.append(x, x[-1] + 1.0)
        y_diff_raw = np.append(y_diff_raw, 0.0)
        # -------------------------

        y_diff_smooth = smooth_savgol(y_diff_raw)

        # Decomposition is performed on the smoothed signal
        fit_result = decompose_survival_difference(x, y_diff_smooth, label=title)

        log.info(f"\n  >>> RESULT: {fit_result['n_pos']}+ / {fit_result['n_neg']}− "
                 f"= {fit_result['n_pos'] + fit_result['n_neg']} total, "
                 f"MSE={fit_result['mse_total']:.4f}")

        all_results.append({
            'title': title,
            'x': x,
            'y_raw': y_diff_raw,
            'y_smooth': y_diff_smooth,
            'fit': fit_result
        })

    # ── Parameter tables ──────────────────────────────────────────────────────
    print("\n" + "="*90)
    print("  GAUSSIAN DECOMPOSITION: PARAMETER ESTIMATES WITH 95% CI")
    print("  BIC-based model selection | Savitzky–Golay smoothed (window=7, poly=2)")
    print("="*90)

    for r in all_results:
        print_parameters_table(r['fit'], r['title'])

    # ── Significant components only ───────────────────────────────────────────
    print("\n" + "="*90)
    print("  SIGNIFICANT COMPONENTS ONLY (for paper)")
    print("  Components where 95% CI for amplitude does NOT include zero")
    print("="*90)

    for r in all_results:
        fit = r['fit']
        title = r['title']
        sig_count = 0
        total_count = fit['n_pos'] + fit['n_neg']
        lines = []

        for sign_label, best, prefix, neg in [('(+)', fit['best_pos'], '+G', False),
                                               ('(−)', fit['best_neg'], '−G', True)]:
            if best is None:
                continue
            for k, comp in enumerate(best['components']):
                if comp['significant']:
                    sig_count += 1
                    amp = -comp['amplitude'] if neg else comp['amplitude']
                    lines.append(
                        f"  {prefix}{k+1:<5} {sign_label:<5} "
                        f"{amp:>11.2f} ± {comp['ci_amplitude']:>6.2f} "
                        f"{comp['mu']:>9.2f} ± {comp['ci_mu']:>5.2f} "
                        f"{comp['sigma']:>8.2f} ± {comp['ci_sigma']:>5.2f}"
                    )

        print(f"\n  {title}")
        print(f"  {sig_count} significant / {total_count} total components")
        print(f"  {'Comp':<7} {'Sign':<5} {'Amplitude':>11} {'± CI':>9} "
              f"{'μ (day)':>9} {'± CI':>8} {'σ':>8} {'± CI':>8}")
        print(f"  {'─'*70}")
        for line in lines:
            print(line)

    # ── Metrics tables (MSE / RMSE / R²) ─────────────────────────────────────
    print("\n\n" + "="*90)
    print("  GAUSSIAN FIT QUALITY: MSE / RMSE / R²")
    print("  R² = 1 - MSE / Var(y)  |  >0.95 excellent | >0.90 good | <0.80 review")
    print("  (s) = vs smoothed signal   (r) = vs raw signal")
    print("="*90)

    summary_rows = []

    for r in all_results:
        metrics = print_metrics_table(
            r['title'],
            r['fit']['y_fit_total'],
            r['y_smooth'],
            r['y_raw']
        )
        summary_rows.append((r['title'], *metrics))

    # ── Summary metrics table (paper-ready) ──────────────────────────────────
    print("\n\n" + "="*90)
    print("  SUMMARY TABLE (for paper)")
    print("="*90)
    print(f"  {'Comparison':<42} {'MSE(s)':>7} {'RMSE(s)':>8} {'R²(s)':>7} "
          f"{'MSE(r)':>7} {'RMSE(r)':>8} {'R²(r)':>7}  Quality(raw)")
    print(f"  {'─'*88}")

    for row in summary_rows:
        name, mse_s, rmse_s, r2_s, mse_r, rmse_r, r2_r = row
        short = name[:40]
        print(f"  {short:<42} {mse_s:>7.4f} {rmse_s:>8.4f} {r2_s:>7.4f} "
              f"{mse_r:>7.4f} {rmse_r:>8.4f} {r2_r:>7.4f}  {r2_quality(r2_r)}")

    # ── Superposition amplitudes at Gaussian centers ──────────────────────────
    print("\n\n" + "="*90)
    print("  SUPERPOSITION AMPLITUDES AT GAUSSIAN CENTERS (μ)")
    print("="*90)

    for r in all_results:
        fit = r['fit']
        print(f"\n  {r['title']}")
        print(f"  {'Comp':<7} {'Center (μ)':>12} | {'Superposition Amp (%)':>22}")
        print(f"  {'─'*47}")

        centers = []
        if fit['best_pos'] is not None:
            for k, comp in enumerate(fit['best_pos']['components']):
                centers.append((f"+G{k+1}", comp['mu']))
        if fit['best_neg'] is not None:
            for k, comp in enumerate(fit['best_neg']['components']):
                centers.append((f"−G{k+1}", comp['mu']))

        # Сортируем по времени (μ)
        centers.sort(key=lambda item: item[1])

        if not centers:
            print("  No components found.")
        else:
            for name, mu in centers:
                total_amp = get_superposition_amplitude(mu, fit)
                print(f"  {name:<7} {mu:>12.2f} | {total_amp:>22.2f}")

    print()
    print("="*90)

    # ── BIC curves ────────────────────────────────────────────────────────────
    fig_bic, axes_bic = plt.subplots(2, 2, figsize=(16, 10))
    for i, r in enumerate(all_results):
        plot_bic_curves(axes_bic.flatten()[i], r['fit'], r['title'])
    fig_bic.suptitle('BIC vs Number of Gaussians (model selection)',
                     fontweight='bold', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.savefig('bic_curves.png', dpi=600, bbox_inches='tight')
    plt.show()

    # ── Main decomposition plots ──────────────────────────────────────────────
    fig_main, axes_main = plt.subplots(2, 2, figsize=(20, 14))
    for i, r in enumerate(all_results):
        plot_decomposition(
            axes_main.flatten()[i],
            r['x'], r['y_raw'], r['y_smooth'],
            r['fit'], r['title']
        )
    fig_main.suptitle(
        'Gaussian Decomposition of Survival Differences\n'
        '(Raw data points · Smoothed dashed · BIC-selected Gaussian fit)',
        fontweight='bold', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.savefig('decomposition_smooth.png', dpi=600, bbox_inches='tight')
    plt.show()

    return all_results


# ============================================================
# RUN
# ============================================================
if __name__ == '__main__':
    results = main()

In [ ]:
import pandas as pd
import numpy as np

# ==============================
# НАСТРОЙКИ
# ==============================
INPUT_FILE  = "/content/drive/MyDrive/Полином/задание (1).xlsx"
OUTPUT_FILE = "survival_differences.xlsx"

# ==============================
# ЗАГРУЗКА ДАННЫХ
# ==============================
def load_data(path):
    df_raw = pd.read_excel(path)

    df = df_raw.iloc[3:, :].reset_index(drop=True)
    df = df.iloc[:, 1:]

    df.columns = [
        'exp_dead_f', 'exp_dead_m', 'exp_alive_f', 'exp_alive_m',
        'ctrl_dead_f', 'ctrl_dead_m', 'ctrl_alive_f', 'ctrl_alive_m'
    ]

    def to_float(x):
        if pd.isna(x) or x == '':
            return 0.0
        if isinstance(x, str):
            x = x.replace(',', '.')
        return float(x)

    for col in ['exp_alive_f', 'exp_alive_m', 'ctrl_alive_f', 'ctrl_alive_m']:
        df[col] = df[col].apply(to_float)

    df['age'] = np.arange(len(df))
    return df


# ==============================
# ОСНОВНАЯ ЛОГИКА
# ==============================
def compute_differences(df):
    result = pd.DataFrame()

    result['age'] = df['age']

    # Разности
    result['F_exp_vs_ctrl'] = df['exp_alive_f'] - df['ctrl_alive_f']
    result['M_exp_vs_ctrl'] = df['exp_alive_m'] - df['ctrl_alive_m']
    result['Exp_F_vs_M']    = df['exp_alive_f'] - df['exp_alive_m']
    result['Ctrl_F_vs_M']   = df['ctrl_alive_f'] - df['ctrl_alive_m']

    return result


# ==============================
# СОХРАНЕНИЕ В EXCEL
# ==============================
def save_to_excel(df, path):
    df.to_excel(path, index=False)
    print(f"Файл сохранён: {path}")


# ==============================
# RUN
# ==============================
if __name__ == "__main__":
    df = load_data(INPUT_FILE)
    diff_df = compute_differences(df)
    save_to_excel(diff_df, OUTPUT_FILE)

In [ ]:
!pip install lifelines -q

"""
UNIFIED PIPELINE: GAUSSIAN DECOMPOSITION & BOOTSTRAP VALIDATION
================================================================================
1. Exact Gaussian decomposition of survival differences (ΔS) with BIC selection.
2. Metrics (MSE/RMSE/R²) and Confidence Intervals (CI).
3. Bootstrap validation using Kaplan-Meier resampling to assess robustness.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from scipy.stats import norm
from scipy.interpolate import interp1d
try:
    from lifelines import KaplanMeierFitter
except ImportError:
    print("Please install lifelines first: pip install lifelines")
    raise
import warnings
import logging

# ============================================================
# 1. SETUP & CONFIGURATION
# ============================================================
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING, format='%(message)s')
log = logging.getLogger(__name__)

# --- ОБНОВЛЕНИЕ: Все шрифты 14, DPI 600 ---
plt.rcParams.update({
    'font.size': 14, 'axes.titlesize': 14, 'axes.labelsize': 14,
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 14,
    'figure.dpi': 600, 'savefig.dpi': 600, 'font.family': 'sans-serif'
})

np.random.seed(42)

# --- Настройки путей и параметров ---
FILE_PATH = "/content/drive/MyDrive/Полином/задание (1).xlsx" # Укажите ваш путь

COMPARISONS = [
    ("Females: Experimental vs Control", 'exp_alive_f', 'ctrl_alive_f'),
    ("Males: Experimental vs Control",   'exp_alive_m', 'ctrl_alive_m'),
    ("Exp Group: Females vs Males",      'exp_alive_f', 'exp_alive_m'),
    ("Ctrl Group: Females vs Males",     'ctrl_alive_f', 'ctrl_alive_m')
]

N_BOOTSTRAP = 100                     # Количество итераций бутстрепа
MAX_GAUSSIANS_PER_HALF = 8            # Максимум гауссиан для подбора
CONFIDENCE_LEVEL = 0.95
Z_CRIT = norm.ppf(1 - (1 - CONFIDENCE_LEVEL) / 2)

SAVGOL_WINDOW = 7
SAVGOL_POLYORDER = 2


# ============================================================
# 2. DATA LOADING & PREPROCESSING
# ============================================================
def load_data(path):
    """Load and parse the survival data from Excel."""
    try:
        df_raw = pd.read_excel(path)
    except FileNotFoundError:
        log.error(f"File not found: {path}")
        return None

    df = df_raw.iloc[3:, :].reset_index(drop=True)
    df = df.iloc[:, 1:]
    df.columns = [
        'exp_dead_f', 'exp_dead_m', 'exp_alive_f', 'exp_alive_m',
        'ctrl_dead_f', 'ctrl_dead_m', 'ctrl_alive_f', 'ctrl_alive_m'
    ]

    def to_float(x):
        try:
            if pd.isna(x) or x == '': return 0.0
            if isinstance(x, str): x = x.replace(',', '.')
            return float(x)
        except (ValueError, TypeError):
            return 0.0

    for col in ['exp_alive_f', 'exp_alive_m', 'ctrl_alive_f', 'ctrl_alive_m']:
        df[col] = df[col].apply(to_float)

    df['age'] = np.arange(len(df))
    return df

def smooth_savgol(y, window_length=SAVGOL_WINDOW, polyorder=SAVGOL_POLYORDER):
    w = min(window_length, len(y))
    if w % 2 == 0: w = max(3, w - 1)
    if w > polyorder: return savgol_filter(y, w, polyorder)
    return y.copy()


# ============================================================
# 3. GAUSSIAN MATH & BIC FITTING
# ============================================================
def single_gaussian(x, amp, mu, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def multi_gaussian(x, *params):
    n = len(params) // 3
    result = np.zeros_like(x, dtype=float)
    for i in range(n):
        result += single_gaussian(x, params[3*i], params[3*i+1], params[3*i+2])
    return result

def compute_bic(n_points, n_params, rss):
    if rss <= 0 or n_points <= n_params: return np.inf
    return n_points * np.log(rss / n_points) + n_params * np.log(n_points)

def guess_initial_params(x, y, n_gauss):
    residual = y.copy()
    params = []
    x_range = x[-1] - x[0]
    for _ in range(n_gauss):
        idx = np.argmax(residual)
        amp = max(residual[idx], 0.01)
        mu = x[idx]
        sigma = max(1.0, x_range / (2 * n_gauss))
        params.extend([amp, mu, sigma])
        residual = residual - single_gaussian(x, amp, mu, sigma)
        residual = np.maximum(residual, 0)
    return params

def fit_n_gaussians(x, y, n_gauss, label=""):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n_points = len(x)
    n_params = 3 * n_gauss
    x_range = x[-1] - x[0] if len(x) > 1 else 1.0

    if np.max(y) < 1e-9: return None

    p0 = guess_initial_params(x, y, n_gauss)
    sigma_min = 1.0
    lower, upper = [], []
    for k in range(n_gauss):
        lower.extend([0.0, x[0] - 1.0, sigma_min])
        upper.extend([np.max(y) * 2.0, x[-1] + 1.0, x_range * 1.5])

    try:
        popt, pcov = curve_fit(multi_gaussian, x, y, p0=p0, bounds=(lower, upper), maxfev=50000)
    except Exception:
        return None

    if np.any(np.isinf(pcov)) or np.any(np.isnan(pcov)): return None

    y_fit = multi_gaussian(x, *popt)
    rss = np.sum((y - y_fit) ** 2)
    mse = rss / n_points
    bic = compute_bic(n_points, n_params, rss)

    std_errors = np.sqrt(np.diag(pcov))
    ci_half = Z_CRIT * std_errors

    components = []
    for k in range(n_gauss):
        amp, mu, sigma = popt[3*k], popt[3*k+1], popt[3*k+2]
        ci_amp = ci_half[3*k]
        significant = (amp - ci_amp) > 0

        components.append({
            'amplitude': amp, 'ci_amplitude': ci_amp,
            'mu': mu, 'ci_mu': ci_half[3*k+1],
            'sigma': sigma, 'ci_sigma': ci_half[3*k+2],
            'significant': significant
        })

    return {
        'n_gauss': n_gauss, 'popt': popt, 'pcov': pcov,
        'y_fit': y_fit, 'mse': mse, 'bic': bic,
        'components': components, 'n_params': n_params
    }

def find_optimal_n_bic(x, y, label="", max_n=MAX_GAUSSIANS_PER_HALF):
    if np.max(y) < 1e-9: return {}, 0

    threshold = np.max(y) * 0.01
    n_informative = np.sum(y > threshold)
    effective_max_n = min(max_n, max(1, n_informative // 6))

    results = {}
    best_bic = np.inf
    best_n = 0

    for n in range(1, effective_max_n + 1):
        res = fit_n_gaussians(x, y, n, label=label)
        if res is None:
            results[n] = None
            continue
        results[n] = res
        if res['bic'] < best_bic:
            best_bic = res['bic']
            best_n = n

    return results, best_n

def decompose_survival_difference(x, y_diff, label=""):
    y_pos = np.maximum(y_diff, 0.0)
    y_neg = np.maximum(-y_diff, 0.0)

    res_pos, n_pos = find_optimal_n_bic(x, y_pos, label="POS")
    res_neg, n_neg = find_optimal_n_bic(x, y_neg, label="NEG")

    best_pos = res_pos.get(n_pos) if n_pos > 0 else None
    y_fit_pos = best_pos['y_fit'] if best_pos else np.zeros_like(x)

    best_neg = res_neg.get(n_neg) if n_neg > 0 else None
    y_fit_neg = best_neg['y_fit'] if best_neg else np.zeros_like(x)

    y_fit_total = y_fit_pos - y_fit_neg
    mse_total = np.mean((y_diff - y_fit_total) ** 2)

    return {
        'n_pos': n_pos, 'n_neg': n_neg,
        'best_pos': best_pos, 'best_neg': best_neg,
        'y_fit_pos': y_fit_pos, 'y_fit_neg': -y_fit_neg,
        'y_fit_total': y_fit_total,
        'mse_total': mse_total,
        'res_pos': res_pos, 'res_neg': res_neg
    }

def compute_metrics(y_true, y_fit):
    mse = np.mean((y_true - y_fit) ** 2)
    rmse = np.sqrt(mse)
    var = np.var(y_true)
    r2 = 1.0 - mse / var if var > 0 else np.nan
    return mse, rmse, r2

def get_superposition_amplitude(t, fit_result):
    val_pos, val_neg = 0.0, 0.0
    if fit_result['best_pos']: val_pos = multi_gaussian(np.array([t]), *fit_result['best_pos']['popt'])[0]
    if fit_result['best_neg']: val_neg = multi_gaussian(np.array([t]), *fit_result['best_neg']['popt'])[0]
    return val_pos - val_neg


# ============================================================
# 4. BOOTSTRAP MATH
# ============================================================
def survival_to_death_probs(survival_pct):
    s = np.array(survival_pct) / 100.0
    n_days = len(s)
    probs = np.zeros(n_days)
    for t in range(1, n_days):
        probs[t] = max(s[t - 1] - s[t], 0)
    probs[-1] += max(s[-1], 0)
    total = probs.sum()
    if total > 0: probs /= total
    return probs

def infer_sample_size(survival_pct):
    s = np.array(survival_pct)
    drops = [s[t - 1] - s[t] for t in range(1, len(s)) if (s[t - 1] - s[t]) > 0.01]
    if not drops: return 100
    return max(round(100.0 / min(drops)), 30)

def resample_and_km(death_probs, n_flies, max_day):
    days_possible = np.arange(len(death_probs))
    death_days = np.random.choice(days_possible, size=n_flies, p=death_probs)
    kmf = KaplanMeierFitter()
    kmf.fit(death_days, event_observed=np.ones(n_flies))
    eval_days = np.arange(max_day + 1, dtype=float)
    survival = np.array([kmf.predict(d) for d in eval_days]) * 100.0
    return eval_days, survival


# ============================================================
# 5. VISUALIZATION FUNCTIONS
# ============================================================
def plot_decomposition(ax, x, y_raw, y_smooth, fit_result, title):
    n_pos, n_neg = fit_result['n_pos'], fit_result['n_neg']
    mse_s, rmse_s, r2_s = compute_metrics(y_smooth, fit_result['y_fit_total'])
    mse_r, rmse_r, r2_r = compute_metrics(y_raw,    fit_result['y_fit_total'])

    x_dense = np.linspace(x[0], x[-1], 1000)
    ax.plot(x, y_raw, 'o', color='#555555', markersize=5, alpha=0.55, label='ΔS(t) raw', zorder=5)

    f_total = interp1d(x, fit_result['y_fit_total'], kind='cubic', bounds_error=False, fill_value='extrapolate')
    ax.plot(x_dense, f_total(x_dense), '-', color='#e6005c', linewidth=2.5,
            label=f'Total fit  R²(smooth)={r2_s:.3f}  R²(raw)={r2_r:.3f}', zorder=6)

    # Plot individuals
    mus = []
    if fit_result['best_pos']:
        colors_p = plt.cm.Blues(np.linspace(0.4, 0.85, max(n_pos, 1)))
        popt = fit_result['best_pos']['popt']
        for k in range(n_pos):
            amp, mu, sigma = popt[3*k], popt[3*k+1], popt[3*k+2]
            y_k = single_gaussian(x_dense, amp, mu, sigma)
            ax.plot(x_dense, y_k, '-', color=colors_p[k], linewidth=1.5, label=f'+G{k+1}')
            ax.fill_between(x_dense, 0, y_k, color=colors_p[k], alpha=0.12)
            mus.append(mu)

    if fit_result['best_neg']:
        colors_n = plt.cm.Reds(np.linspace(0.4, 0.85, max(n_neg, 1)))
        popt = fit_result['best_neg']['popt']
        for k in range(n_neg):
            amp, mu, sigma = popt[3*k], popt[3*k+1], popt[3*k+2]
            y_k = -single_gaussian(x_dense, amp, mu, sigma)
            ax.plot(x_dense, y_k, '-', color=colors_n[k], linewidth=1.5, label=f'−G{k+1}')
            ax.fill_between(x_dense, 0, y_k, color=colors_n[k], alpha=0.12)
            mus.append(mu)

    for mu in mus:
        amp = get_superposition_amplitude(mu, fit_result)
        ax.vlines(x=mu, ymin=0, ymax=amp, color='black', linestyle='-', linewidth=1.5, zorder=7)
        ax.plot(mu, amp, 'o', color='black', markersize=6, zorder=8)

    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.set_title(f'{title}\n({n_pos}+ / {n_neg}− = {n_pos+n_neg} components)', fontweight='bold')
    ax.set_xlabel('Age (days)')
    ax.set_ylabel('Survival Difference (%)')
    ax.legend(loc='best', fontsize=12, ncol=2)

def plot_bic_curves(ax, fit_result, title):
    for res_dict, n_opt, color, label in [
        (fit_result['res_pos'], fit_result['n_pos'], '#2196F3', 'Positive'),
        (fit_result['res_neg'], fit_result['n_neg'], '#F44336', 'Negative')
    ]:
        ns = sorted([n for n in res_dict if n > 0 and res_dict[n] is not None])
        if not ns: continue
        bics = [res_dict[n]['bic'] for n in ns]
        ax.plot(ns, bics, 'o-', color=color, lw=2, markersize=6, label=f'{label} (opt={n_opt})')
        ax.axvline(n_opt, color=color, linestyle='--', alpha=0.5)

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Number of Gaussians')
    ax.set_ylabel('BIC')
    ax.grid(True, ls=':', alpha=0.5)
    ax.legend()


# ============================================================
# 6. MAIN EXECUTION PIPELINE
# ============================================================
def main():
    print("="*80)
    print(" 1. LOADING DATA AND EXACT DECOMPOSITION")
    print("="*80)

    df = load_data(FILE_PATH)
    if df is None: return

    all_results = []

    # ─── PART A: EXACT FIT ──────────────────────────────────────────────
    for title, col1, col2 in COMPARISONS:
        mask = (df[col1] > 0) | (df[col2] > 0)
        last_idx = df[mask].index[-1]
        x = df['age'].values[:last_idx + 1].astype(float)

        y_diff_raw = (df[col1].values[:last_idx + 1] - df[col2].values[:last_idx + 1]).astype(float)

        # Замыкание на ноль
        x = np.append(x, x[-1] + 1.0)
        y_diff_raw = np.append(y_diff_raw, 0.0)

        y_diff_smooth = smooth_savgol(y_diff_raw)
        fit_result = decompose_survival_difference(x, y_diff_smooth, label=title)

        all_results.append({
            'title': title, 'col1': col1, 'col2': col2,
            'x': x, 'y_raw': y_diff_raw, 'y_smooth': y_diff_smooth,
            'fit': fit_result
        })

    # Plot exact fits
    fig_main, axes_main = plt.subplots(2, 2, figsize=(20, 14))
    for i, r in enumerate(all_results):
        plot_decomposition(axes_main.flatten()[i], r['x'], r['y_raw'], r['y_smooth'], r['fit'], r['title'])
    fig_main.suptitle('Gaussian Decomposition of Survival Differences (Exact)', fontweight='bold', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.savefig('decomposition_exact.png', dpi=600, bbox_inches='tight')
    plt.show()

    # ─── PART B: BOOTSTRAP VALIDATION ────────────────────────────────────
    print("\n" + "="*80)
    print(f" 2. RUNNING BOOTSTRAP VALIDATION (N={N_BOOTSTRAP})")
    print("="*80)

    max_day = len(df['age']) - 1
    bootstrap_results = {}

    # СОЗДАЕМ ПОСТЕР ДЛЯ ВСЕХ ГРУПП (Строки = группы, Столбцы = 3 параметра)
    n_groups = len(all_results)
    fig_poster, axes_poster = plt.subplots(n_groups, 3, figsize=(20, 6 * n_groups))

    for i, r in enumerate(all_results):
        title, col1, col2 = r['title'], r['col1'], r['col2']
        print(f"\nBootstrapping: {title} ...")

        dist1 = survival_to_death_probs(df[col1])
        dist2 = survival_to_death_probs(df[col2])
        n1 = infer_sample_size(df[col1])
        n2 = infer_sample_size(df[col2])

        components = []
        for b in range(N_BOOTSTRAP):
            x1, s1 = resample_and_km(dist1, n1, max_day)
            x2, s2 = resample_and_km(dist2, n2, max_day)

            min_len = min(len(s1), len(s2))
            x_b = np.arange(min_len, dtype=float)
            dS_smooth = smooth_savgol(s1[:min_len] - s2[:min_len])

            res = decompose_survival_difference(x_b, dS_smooth)

            for sign, best, is_neg in [('pos', res['best_pos'], False), ('neg', res['best_neg'], True)]:
                if best is None: continue
                for comp in best['components']:
                    if comp['significant']:
                        components.append({
                            'bootstrap': b, 'sign': sign,
                            'amplitude': -comp['amplitude'] if is_neg else comp['amplitude'],
                            'mu': comp['mu'], 'sigma': comp['sigma']
                        })
            if (b + 1) % 25 == 0: print(f"  ... {b+1}/{N_BOOTSTRAP} done")

        df_comp = pd.DataFrame(components)
        bootstrap_results[title] = df_comp

        # ДОБАВЛЯЕМ ГРАФИКИ НА ПОСТЕР
        if len(df_comp) > 0:
            ax_mu, ax_sig, ax_amp = axes_poster[i, 0], axes_poster[i, 1], axes_poster[i, 2]

            neg = df_comp[df_comp['sign'] == 'neg']
            pos = df_comp[df_comp['sign'] == 'pos']

            # Собираем данные в списки, чтобы столбцы не сливались (dodge/side-by-side)
            mu_data = []
            mu_colors = []
            mu_labels = []
            if len(neg) > 0:
                mu_data.append(neg['mu'])
                mu_colors.append('#D32F2F')
                mu_labels.append('Negative')
            if len(pos) > 0:
                mu_data.append(pos['mu'])
                mu_colors.append('#1976D2')
                mu_labels.append('Positive')

            # Параметр rwidth=0.85 делает зазоры между столбиками
            if mu_data:
                ax_mu.hist(mu_data, bins=np.arange(0, max_day+2, 1), color=mu_colors,
                           alpha=0.8, label=mu_labels, rwidth=0.85)

            ax_mu.set(xlabel='μ (day)', title=f'{title}\nA. Component Locations (μ)')
            if mu_labels: ax_mu.legend()
            ax_mu.grid(True, ls=':', alpha=0.4)

            ax_sig.hist(df_comp['sigma'], bins=25, color='#FF9800', alpha=0.8, rwidth=0.85)
            ax_sig.set(xlabel='σ (days)', title='B. Width of components')
            ax_sig.grid(True, ls=':', alpha=0.4)

            ax_amp.hist(df_comp['amplitude'], bins=25, color='#4CAF50', alpha=0.8, rwidth=0.85)
            ax_amp.set(xlabel='Amplitude (%)', title='C. Amplitude of components')
            ax_amp.grid(True, ls=':', alpha=0.4)

    # Завершение работы с постером
    fig_poster.suptitle('Bootstrap Distributions Pipeline (All Comparisons)', fontweight='bold', fontsize=18, y=1.01)
    plt.tight_layout()
    plt.savefig('boot_dist_poster.png', dpi=600, bbox_inches='tight')
    plt.show()

    # Plot Bootstrap Component Counts
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    for ax, (title, df_comp) in zip(axes.flatten(), bootstrap_results.items()):
        if len(df_comp) == 0: continue
        counts = df_comp.groupby('bootstrap').size()
        # Добавляем rwidth для разделения столбцов здесь тоже
        ax.hist(counts, bins=range(counts.min(), counts.max() + 2), color='#5C6BC0',
                align='left', edgecolor='white', rwidth=0.85)
        ax.axvline(counts.mean(), color='#D32F2F', ls='--', lw=2, label=f'mean = {counts.mean():.1f} ± {counts.std():.1f}')
        ax.set(xlabel='Number of components per bootstrap', title=title)
        ax.legend(); ax.grid(True, ls=':', alpha=0.4)
    plt.suptitle('Distribution of the Number of Gaussian Components Found', fontweight='bold', fontsize=18, y=1.02)
    plt.tight_layout()
    plt.savefig('boot_counts.png', dpi=600, bbox_inches='tight')
    plt.show()

    # ─── PART C: TABLES & ROBUSTNESS REPORT ──────────────────────────────
    print("\n" + "="*80)
    print(" 3. FINAL ROBUSTNESS REPORT")
    print("="*80)

    for title, df_comp in bootstrap_results.items():
        print(f"\n >>> {title}")
        for sign_label, sign_val in [("NEGATIVE (-)", "neg"), ("POSITIVE (+)", "pos")]:
            sub = df_comp[df_comp['sign'] == sign_val]
            if len(sub) == 0:
                print(f"   {sign_label}: None found consistently.")
                continue

            mus = sub['mu'].values
            used = np.zeros(len(mus), dtype=bool)
            clusters = []

            for i in range(len(mus)):
                if used[i]: continue
                cluster = [i]
                for j in range(i + 1, len(mus)):
                    if not used[j] and abs(mus[j] - mus[i]) < 3.0: # 3 days threshold for clustering
                        cluster.append(j)
                        used[j] = True
                used[i] = True
                clusters.append(cluster)

            print(f"   {sign_label} Clusters:")
            print(f"   {'Cluster':<8} {'μ mean':<9} {'μ std':<7} {'Amp mean':<10} {'σ mean':<8} {'Robust%':<8}")
            print(f"   {'-'*55}")

            for ci, idx_list in enumerate(clusters):
                s = sub.iloc[idx_list]
                n_boots = s['bootstrap'].nunique()
                if n_boots >= 5:  # Only show robust clusters (>= 5% of bootstraps)
                    print(f"   {ci+1:<8} {s['mu'].mean():<9.1f} {s['mu'].std():<7.1f} "
                          f"{s['amplitude'].mean():<10.2f} {s['sigma'].mean():<8.1f} "
                          f"{n_boots/N_BOOTSTRAP*100:<8.0f}%")

if __name__ == '__main__':
    main()